# CSC 421 - Knowledge and Reasoning 1
#### **Propositional logic**

### Instructor: Shengyao Lu


## Recall: problem-solving agents (e.g., informed search, local search on CSP): 
* atomic representations: 
    * domain-specific knowledge (i.e., a specific action from a specific state leads to a specific resulting state)
    * no general facts (e.g., road length can't be negative). 
    * require listing all possible concrete states.
* factored representations: 
    * states are represented as assignments of values to variables.
    * some parts of the agents can work in a domain-independent way.

Next, we will introduce **logic**, as a general class of representations to support knowledge-based agents. 
* For this topic, we will introduce two main concepts:
    * **propositional logic**: uses factored representation, less expressive than FOL.
    * **first-order logic (FOL)**: uses canonical structured representation.

1. [KB agents](#1-knowledge-based-agents)
2. [Propositional logic](#2-propositional-logic)
3. [Propositional theorem proving](#3-propositional-theorem-proving)
    - [Proof by inference rules](#31-inference-rules-to-derive-a-proof)
    - [Proof by resolution](#32-proof-by-resolution)


## READINGS  
The section numbers are based on the 4th edition of the textbook. 
They also should work for the 3rd edition. 

CHAPTER 7 LOGICAL AGENTS 

* Basic: Sections 7.1, 7.2, 7.3, 7.4, and Summary
* Expected: 7.5 
* Advanced: All the chapter including bibliographical and historical notes

## 1. Knowledge-based agents

* are the agents that use a process of **reasoning** over an **internal representation of knowledge** to decide what actions to take. 
* they can **combine** and **recombine** information to suit various purposes.



### Terminology
- **Knowledge Base (KB)** := a set of "sentences" known by a KB agent. 
- **sentence**: is an assertion about the world in a knowledge representation language. 
- **axiom**: sentence that holds without deriving from other other sentences.  
- **inference**: deriving new sentences from old.
    - TELL: add new sentences to the knowledge base.
    - ASK: query what is known.

<img src="images/kb-pseudo.png" width="500px"> <br></br>
Each time the agent program is called, it does three things: 

1. TELLs the KB what it perceives
2. ASKs the KB what action it should perform (in the process of answering this query, reasoning is performed about the current state of the world, outcomes of possible action sequences, etc.)
3. TELLs the KB what action was chosen and then the agent executes the action

### The Wumpus World

<img src="images/wumpus_world.png" width="600px">

Wumpus can be shot but agent only has one arrow. Some rooms contain bottomless pits. One location contains gold. 

* **Performance measure:** 
    - +1000 for climbing out of the cave with the gold, 
    - -1000 for falling into a pit or being eaten by wumpus, 
    - -1 for each action taken, 
    - -10 for using the arrrow. 
    - The game ends when the agent dies or when the agent climbs out of the cave.
* **Environment:** A $4 \times 4$ grid of rooms. The agent always starts in the square labeled $[1,1]$ facing to the right. The locations of the gold and wumpus are chosen randomly with a uniform distribution. Each square other than the start can be a pit, with probability 0.2.
* **Actuators:** 
    - The agent can move *Forward*, *TurnLeft* by $90^o$, or *TurnRight* by $90^o$. The agent dies a horrible death if it enters a square with a pit or wumpus (unless the wumpus is dead). If an agent tries to move forward and bumps into wall, it stays in the same square. 
    - The action *Grab* can be used to pick up the gold if it is in the same square as the agent. 
    - The action *Shoot* can be used to fire an arrow in a stright line in the direction the agent is facing. The arrow continues until it either hits (and kills) the Wumpus or hits a wall. The agent has only one arrow. 
    - The action *Climb* can be used to climb out of the cave, but only from square (1,1). 
* **Sensors:** The agent has five sensors, each of which gives a single bit of information:
    * In the square containing the wumpus and directly adjacent (not diagonally) squares, the agent with perceive a *Stench*.
    * In the squares directly adjacent to a pit, the agent will perceive a *Breeze*
    * In the square where the gold is, the agent will perceive a *Glitter*
    * When the agent walk into a wall, it will perceive a *Bump*
    * When the wumpus is killed, it emits a woeful *Scream* that can be perceived from anywhere in the cave 

<img src="images/wumpus_01.png" width="800px">

(see demo)

### Logic

- **Syntax**: the stucture of sentences. It specifies whether the sentences are well-formed, 
    - e.g., $x+4=y$ is well-formed, 
    - $x4y+=$ is not.
- **Semantics**: the way where the truth of sentences is determined. They define the truth of each sentence w.r.t. each world in a problem, 
    - e.g., $x+4=y$ is true for $x=1,y=5$, 
    - $x+4=y$ is false for $x=5,y=1$.
- A **model** is a possible world in which every sentence has a truth value.
    - e.g. in the world $m$ that seed=12345678, Pit(2,3) = True, Pit(1,2) = False, Breeze(1,1) = False. 
    - Here, the sentence $\alpha=$"Pit(2,3)" satisfies model $m$ as it is True in this world. 
    - $M(\alpha)=\{m,n,\dots\}$ is the set of all models where $\alpha$ satisfies.
- **Logical entailment:** one sentence follows logically from another in every model.
    - two sentences $\alpha,\beta$, an entailment is: $\alpha \vDash \beta$ if and only if, in every model in which $\alpha$ is true, $\beta$ is also true. 
    - a.k.a. $\alpha \vDash \beta$ if and only if $M(\alpha)\subseteq M(\beta).$  
- **Logical inference:** the process of deriving new sentences from old ones. 
- **Model checking:** 
    - To determine if $KB\vDash \alpha$:
        - Enumerate all possible model.
        - if $M(KB)\subseteq M(\alpha)$ then $KB$ entails $\alpha$.
- **Inference algorithm** $i$ derives sentence $\alpha$ from KB: $KB\vdash_i \alpha.$
    - **soundness**: an inference algorithm is sound when it's truth-preserving. Such algorithm only derives sentences entailed from KB. 
    - **completeness**: an inference algorithm is complete if it can derive any sentence that is entailed from KB. 
    - model checking is an inference algorithm. It is sound and complete, but computationally intractable in many real world cases. 
- **Grounding**: connects logical reasoning processes to the real env where the agent exists. 
- **Learning**: when general rules are produced by a sentence construction process. 
        

## 2. Propositional logic

### Terminology
- **proposition symbol:** $P, Q, R$, etc.
- **atomic sentences:** consist of a single proposition symbol.
- **complex sentences:** constructed from simpler sentences with logical connectives.
- logical connectives:
    - $\neg$ not, negation, e.g., $\neg P$.
    - $\wedge$ and, conjunction, e.g., $P\wedge Q$.
    - $\vee$ or, disjunction, e.g., $P\vee Q$.
    - $\Rightarrow$ implies, implication, e.g., $P\Rightarrow Q$.
    - $\Leftrightarrow$ if and only if, biconditional, e.g., $P\Leftrightarrow Q$.
- **literal:** is either positive–an atomic sentence $P$ or negative–a negated atomic sentence $\neg P$. 
- Truth tables: describe the semantics of propositional logic. 


<img src="images/truth-table-pl.png" width="800px">


### Example of model checking in propositional logic
- $P:$ It's a Tuesday.
- $Q:$ It's raining.
- $R:$ Harry will go out. 
- $KB: (P\vee Q)\rightarrow R, P, \neg Q$
    - a.k.a. $((P\vee Q)\rightarrow R)\wedge P\wedge \neg Q$

**❓Query**: $R$

By doing model checking, we aim to figure out: given the info in $KB$, do we know for sure that $R$ holds?

To determine whether $KB\vDash R$, we should list all models. 

| P     | Q     | R     | KB |
|-------|-------|-------|----|
| $False$ | $False$ | $False$ |    |
| $False$ | $False$ | $True$  |    |
| $False$ | $True$  | $False$ |    |
| $False$ | $True$  | $True$  |    |
| $True$  | $False$ | $False$ |    |
| $True$  | $False$ | $True$  |    |
| $True$  | $True$  | $False$ |    |
| $True$  | $True$  | $True$  |    |

| P     | Q     | R     | KB    |
|-------|-------|-------|-------|
| $False$ | $False$ | $False$ | $False$ |
| $False$ | $False$ | $True$  | $False$ |
| $False$ | $True$  | $False$ | $False$ |
| $False$ | $True$  | $True$  | $False$ |
| $True$  | $False$ | $False$ | $False$ |
| **True**  | **False** | **True**  | **True**  |
| $True$  | $True$  | $False$ | $False$ |
| $True$  | $True$  | $True$  | $False$ |

Since when $KB$ is true, $R$ is true, we have $M(KB)\subseteq M(R)$. Therefore, yes, $KB\vDash R.$

### Knowledge base in Wumpus world

* Let $P_{ij}$ be true if there is a pit in $[i,j]$.
* Let $W_{ij}$ be true if there is a wumpus in $[i,j]$, dead or alive.
* Let $B_{ij}$ be true if there is breeze in $[i,j]$.
* Let $S_{ij}$ be true if there is stench in $[i,j]$.
* Let $L_{ij}$ be true if the agent is in location $[i,j]$.



1. How can you express the sentence “Pits cause breeze in adjacent squares” ? 

    $B_{11} \Leftrightarrow (P_{12} \lor P_{21})$ <br> 
    $B_{21} \Leftrightarrow (P_{11} \lor P_{22} \lor P_{31})$ <br> 
    $\dots$

    Notice that a single sentence in English may have to be expanded to several propositional logic sentences. 
    In propositional logic we need to state that "pits cause breeze in adjacent square" specifically for every single pit and square. 

2. Now consider a KB consist of the following sentences:
    - There is no pit in $[1,1]$: <br> 
        $R_1:\neg P_{11}$
    - A square is breezy if and only if there is a pit in a neighboring square. This has to be stated for each square; for now, we include just the relevant squares: <br> 
        $R_2: B_{11} \Leftrightarrow (P_{12} \lor P_{21})$ <br> 
        $R_3: B_{21} \Leftrightarrow (P_{11} \lor P_{22} \lor P_{31})$ <br> 
    - Given the knowledge about breeze that percieve in Fig 7.3b: <br>
        $R_4: \neg B_{11}$ <br>
        $R_5: B_{21}$
    - Then we have $KB: R_1\wedge R_2\wedge R_3\wedge R_4\wedge R_5.$

## 3. Propositional theorem proving

We have been using _model checking_ to determine entailment. However, it's costly.

### Terminology 
- **Theorem proving:** applying **rules** of inference directly to the sentences in our KB to construct a proof of the query. 
    - can sometimes more efficient than model checking.
- **Logical equivalence:** two sentences are logically equivalent if they are true in the same set of models. 
    - $\alpha \equiv \beta$ if and only if $\alpha\vDash \beta$ and $\beta\vDash\alpha$. <br><br>
    <img src="images/logic_equiv.png" width="600px"><br><br>
- **Validity**. A sentence is valid if it is true in all models, which a.k.a. a tautology.
    - $P\vee \neg P$
- **Satisfiability**. A sentence is satisfiable if it is true in, or satisfied by, some model.
- **Deduction theorem:** $\alpha\vDash\beta$ if and only if $(\alpha\Rightarrow\beta)$ is valid.
- **Proof by contradiction (reduction ad absurdum)**: $\alpha \vDash \beta$ if and only if $(\alpha \wedge \neg \beta)$ is unsatisfiable. 

### Exercise

Have a KB consisting 3-4 variables sometimes directly specified sometimes described in human language 
and a sentence that you want to prove is entailed by the KB. 

1) Show it by model checking that the models in which the KB is true are a subset of the models in which the sentence is true
2) Show it by the deduction theorem as a single expression with implication and showing that it is true in all possible models 
3) Show it by using proof by contradiction 

Consider three propositional logic boolean variables $A$, $B$, and $C$.Show that the sentence $A \lor (B \land C)$ 
entails the sentence $(A \lor B)$ using model checking, the deduction theorem, and by contradiction. 

Let's make this concrete by assuming that $A$ is I like basketball, $B$ is I am a professor and $C$ is I am in Greece. In that case the entailment could be expressed as if I like basketball or I am a professor and I am in Greece then 
I like basketball or I am a professor. 

|Row| $$A$$| $$B$$ | $$C$$ | $$B \land C$$ | $$A \lor (B \land C)$$   | $$A \lor B$$  |
|:-:|:----:|:-----:|:-----:|:-------------:|:------------------------:|:-------------:|
| 1 | F    | F     | F     | F             | F                        | F             |
| 2 | F    | F     | T     | F             | F                        | F             |
| 3 | F    | T     | F     | F             | F                        | T             |
| 4 | F    | T     | T     | T             | T                        | T             |
| 5 | T    | F     | F     | F             | T                        | T             |                                         
| 6 | T    | F     | T     | F             | T                        | T             |
| 7 | T    | T     | F     | F             | T                        | T             |
| 8 | T    | T     | T     | T             | T                        | T             |

To show this entailment using model checking we need to show that the set of world in which the premise is true is a subset of the worlds in which the premise is true. From the truth table we can see that the premise is true for the worlds that correspond to the rows 4,5,6,7,8 which is a subset of the worlds in which the premise is true 3,4,5,6,7,8. The reverse direction is not true i.e $A \lor B$ does NOT entail $A \lor (B \land C)$ because the worlds in which $A \lor B$ is true are 3,4,5,6,7,8 which is NOT a subset of the worlds in whcih $A \lor (B \land C)$ is true which are 4,5,6,7,8. 

Let's now show entailment using the deduction theorem. 

|Row| $$A$$| $$B$$ | $$C$$ | $$B \land C$$ | $$A \lor (B \land C)$$ |$$A \lor B$$|$$A \lor (B \land C) \rightarrow$$ $$A \lor B  $$| 
|:-:|:----:|:-----:|:-----:|:-------------:|:----------------------:|:---------:|:------------------------------------------:|
| 1 | F    | F     | F     | F             | F                      | F         |  T                                         |
| 2 | F    | F     | T     | F             | F                      | F         |  T                                         |
| 3 | F    | T     | F     | F             | F                      | T         |  T                                         |
| 4 | F    | T     | T     | T             | T                      | T         |  T                                         |
| 5 | T    | F     | F     | F             | T                      | T         |  T                                         |
| 6 | T    | F     | T     | F             | T                      | T         |  T                                         |
| 7 | T    | T     | F     | F             | T                      | T         |  T                                         |
| 8 | T    | T     | T     | T             | T                      | T         |  T                                         |

The last column is calculated by using the truth table for implication and based on the previous two columns corresponding to the premise and consequent. 

Finally let's show entailment using proof by contradiction. In this case we need to show that the sentence: 
$(A \lor (B \land C)) \land \neg (A \lor B)$ is False in all possible worlds. 


|Row| $$A$$| $$B$$ | $$C$$ | $$B \land C$$ | $$A \lor (B \land C)$$ |$$ \neg (A \lor B)$$|$$(A \lor (B \land C)) \land $$ $$\neg (A \lor B)  $$| 
|:-:|:----:|:-----:|:-----:|:-------------:|:----------------------:|:---------:|:------------------------------------------:|
| 1 | F    | F     | F     | F             | F                      | T         |  F                                         |
| 2 | F    | F     | T     | F             | F                      | T         |  F                                         |
| 3 | F    | T     | F     | F             | F                      | F         |  F                                         |
| 4 | F    | T     | T     | T             | T                      | F         |  F                                         |
| 5 | T    | F     | F     | F             | T                      | F         |  F                                         |
| 6 | T    | F     | T     | F             | T                      | F         |  F                                         |
| 7 | T    | T     | F     | F             | T                      | F         |  F                                         |
| 8 | T    | T     | T     | T             | T                      | F         |  F                                         |


### 3.1 Inference rules to derive a proof

| Rule                        | expression                                                                                    | example                                                                                                                                                    |
|-----------------------------|-----------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Modus Ponens                | $$\frac{\alpha\Rightarrow\beta,\alpha}{\beta}$$                                               | $$\frac{\text{If it's raining, then Harry is inside. It is raining.}}{\text{Harry is inside}}$$                                                            |
| And Elimination             | $$\frac{\alpha\wedge\beta}{\alpha}$$                                                          | $$\frac{\text{Harry is friends with Ron and Peter.}}{\text{Harry is friends with Peter.}}$$                                                                |
| Double Negation Elimination | $$\frac{\neg(\neg\alpha)}{\alpha}$$                                                           | $$\frac{\text{It is not true that Harry did not pass the test.}}{\text{Harry passed the test.}}$$                                                          |
| Implication Elimination     | $$\frac{\alpha\Rightarrow\beta}{\neg\alpha\vee\beta}$$                                        | $$\frac{\text{If it is raining, then Harry is inside.}}{\text{It's not raining or Harry is inside.}}$$                                                     |
| Biconditional Elimination   | $$\frac{\alpha\Leftrightarrow\beta}{(\alpha\Rightarrow\beta)\wedge(\beta\Rightarrow\alpha)}$$ | $$\frac{\text{It is raining if and only if Harry is inside.}}{\text{If it is raining, then Harry is inside,and if Harry is inside, then it is raining.}}$$ |
| De Morgan's Law | $$\frac{\neg(\alpha\wedge\beta)}{\neg\alpha \vee\neg\beta}$$ | $$\frac{\text{It is not true that both Harry and Ron passed the test.}}{\text{Harry did not pass the test or Ron didn't pass the test.}}$$ |
| De Morgan's Law | $$\frac{\neg(\alpha\vee\beta)}{\neg\alpha \wedge\neg\beta}$$ | $\dots$ |
| Distributive Property | $$\frac{\alpha\wedge(\beta\vee\gamma)}{(\alpha\vee\beta)\wedge(\alpha\vee\gamma)}$$|$\dots$|
| Distributive Property | $$\frac{(\alpha\vee\beta)\wedge(\alpha\vee\gamma)}{\alpha\wedge(\beta\vee\gamma)}$$|$\dots$|
|$\dots$                     |                                                                                   |                         |

#### Wumpus world example

❓**Q:** Use inference rules to perform theorem proving, showing that $\neg P_{12}$ with $KB=R_1\wedge R_2\wedge R_3\wedge R_4\wedge R_5.$

<img src="images/wumpus_thm_proof.png" width="600px">

(see demo)

#### **❓Q: How to use deduction theorem or by contradiction <mark>with inference rules</mark> to prove $$A \lor (B \land C) \vDash A \lor B$$**


### 3.2 Proof by Resolution

Recall, for traditional search problems:
* initial state
* actions
* transition model
* goal test
* path-cost function

Any of the search algorithms in *csc421_slu_02_search.ipynb* can be used to find a sequence of steps that constitutes a proof like above. We just need to define a proof problem as follows:
- initial state: starting knowledge base
- actions: inference rules
- transition model: new knowledge base after inference
- goal test: check statement we're trying to prove
- path cost function: number of steps in proof


The search algorithms themselves are said to be **complete** in the sense that they will find any reachable goal. However, when the available inference rules are inadequate (e.g., some necessary rules are removed), the inference algorithms may not be complete anymore. Therefore, we introduce **resolution**, yielding a complete inference algorithm when coupled with any complete search algorithm. 



#### **Terminologies**
- **Clause**: a disjunction of literals, e.g., $P\lor Q\lor R$.
- **Unit Resolution Rule**: 
    $$\frac{P\lor Q_1 \lor Q_2 \lor\dots\lor Q_n, \quad \neg P}{Q_1 \lor Q_2 \lor \dots \lor Q_n},$$
    where each $Q_i$ or $P$ is a literal. 
    $$e.g., \frac{\begin{matrix}\text{(Peter is a dog)} \lor \text{(Harry is a dog)} \\ \text{Peter is not a dog}\end{matrix}}{\text{Harry is a dog}}$$
- **Full Resolution Rule:**
    $$\frac{P\lor Q_1 \lor Q_2 \lor\dots\lor Q_n, \quad \neg P\lor R_1 \lor R_2 \lor\dots\lor R_m}{Q_1 \lor Q_2 \lor \dots \lor Q_n\lor R_1 \lor R_2 \lor\dots\lor R_m}.$$
    where each $Q_i, R_j$ or $P$ is a literal. 
    $$e.g., \frac{\begin{matrix}\text{(Peter is a dog)} \lor \text{(Harry is a dog)} \\ \text{(Peter is not a dog)}\lor\text{(Carrie is sleeping)}\end{matrix}}{\text{(Harry is a dog)}\lor\text{(Carrie is sleeping)}}$$
- **Conjunctive Normal Form (CNF):** logical sentence that is a conjunction of clauses.
    - e.g., $(A\lor B\lor C)\land(D\lor\neg E)\land(F\lor G).$  

#### **Conversion to CNF**
- Eliminate biconditionals
    - turn $\alpha\leftrightarrow\beta$ into $(\alpha\rightarrow\beta)\land(\beta\rightarrow\alpha)$
- Eliminate implications
    - turn $\alpha\rightarrow\beta$ into $\neg\alpha\lor\beta$
- Move $\neg$ inwards using De Morgan's Laws
    - e.g., turn $\neg(\alpha\land\beta)$ into $\neg\alpha\lor\neg\beta$
- Use distributive law to distribute $\lor$ wherever possible


#### **A resolution algorithm**

- Recall: To determine if $KB\vDash\alpha$ (proof by contradiction):
    - Check if $(KB\land\neg\alpha)$ is a contradiction?
        - if so, then $KB\vDash\alpha$.
        - otherwise, no entailment.
- To determine if $KB\vDash\alpha$ (proof by contradiction with resolution):
    - convert $(KB\land\neg\alpha)$ to CNF.
    - keep checking to see if we can use resolution to produce a new clause.
        - if ever we produce the empty clause (equiv to _False_), we have a contradiction, and $KB\vDash\alpha$.
        - otherwise, if we can't add new clauses, no entailment.
<br></br>

<img src="images/resolution_psuedo.png" width="600px">

(see demo for $A \lor (B \land C) \vDash A \lor B$)

**❓Q: Does $(A\lor B)\land(\neg B\lor C)\land (\neg C)$ entail $A$?**

#### **Horn clauses and definite clauses**
- **Definite clause:** a disjunction of literals where **exactly one** is positive.
    - e.g., $\neg L_{1,1}\lor\neg Breeze\lor B_{1,1}$
- **Horn clause:** a disjunction of literals where **at most one** is positive.

<br>
 <img src="images/cnf.png" width="700px">

## Summary

How to determine if $KB\vDash A$? 


| Algorithm | model checking | deduction theorem | proof by contradiction |
| - | - | - | - |
| What to check | $M(KB)\subseteq M(A)$? | $(KB\rightarrow A)$ always true? | $(KB\land\neg A)$ always false?|

<br>

| | Truth table | Inference rules | Resolution |
| - | - | - | - |
|How?| $m$ variable, $2^m$ models | Modus Ponens, AND elimination, $\dots$ | convert to CNF, resolution rule |
|enumeration?| ✅ |||
|searchable?|  |✅|✅|